In [1]:
import json
import re
import os
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import spacy

import nltk
nltk.download('wordnet', quiet=True)
from nltk.corpus import wordnet as wn

In [2]:
import sys
print(sys.executable)

/mnt/c/Users/camar/PycharmProjects/F-ATOMIC/.venv/bin/python


In [ ]:
BASE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "pyproject.toml").is_file() and (p / "code/core/generator.py").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")
f_atomic_path = BASE_DIR / "experiments" / "events" / "results" / "forensic_atomic_final.csv"
objects_path = BASE_DIR / "experiments" / "objects" / "results"
results_path = BASE_DIR / "experiments" / "objects" / "results"
objects_path.mkdir(parents=True, exist_ok=True)
results_path.mkdir(parents=True, exist_ok=True)
final_entity_path = objects_path / "entity_classification.csv"

META_FIELDS = ["event"]
SPACY_MODEL = "en_core_web_sm"
SHARE_THRESHOLD = 0.3

# --- Pre-processing degli eventi ATOMIC -------------------------------------
# I placeholder ATOMIC (PersonX, PersonY, PersonZ, ___) confondono il parser:
# li sostituiamo con parole neutre e poi ignoriamo i token corrispondenti.
PERSON_MAP = {"PersonX": "Alex", "PersonY": "Sam", "PersonZ": "Max"}
PERSON_NAMES = set(PERSON_MAP.values())
BLANK = "something"  # per "___"

# --- Regole di filtro strutturale (dichiarate, versionate) -------------------
MAX_TOKENS = 2                      # Fase 1, punto 2
# nomi che non sono entità fisiche in nessun senso utile: persone/ruoli generici,
# astratti molto frequenti negli eventi ATOMIC, pronomi nominali.
STOP_HEADS = {
    "something", "someone", "somebody", "anything", "everything", "thing",
    "one", "way", "time", "day", "lot", "bit", "kind", "sort", "part",
    "place",  # troppo generico; i luoghi specifici (bank, kitchen) restano
    "people", "person", "man", "woman", "boy", "girl", "kid", "child",
    "friend", "mother", "mom", "father", "dad", "sister", "brother", "wife",
    "husband", "boyfriend", "girlfriend", "family", "parent", "son", "daughter",
    "idea", "chance", "fun", "life", "mind", "heart", "love", "turn", "case",
    "attention", "advantage", "sense", "point", "step", "effort", "credit",
}
# lemmi di eventi tipo "get/take/make X" non danno informazione: li teniamo
# solo come contesto (verbo reggente), non come filtro.


# categorie WordNet considerate "fisiche" (oggetto, sostanza, cibo)
PHYS = {"noun.artifact", "noun.substance", "noun.object", "noun.food"}
# categorie ambigue: spesso fisiche nei nostri eventi ma non per WordNet
# (money/credit_card = possession, home = location, hand = body, ...)
AMBIGUOUS = {"noun.possession", "noun.body", "noun.group",
             "noun.plant", "noun.communication"}

In [4]:
df = pd.read_csv(f_atomic_path, usecols=["original_row_id", "event"])
events = df.drop_duplicates("event")[["event"]].reset_index(drop=True)
funnel = {"event_rows": int(len(df)), "unique_events": int(len(events))}

In [5]:
def preprocess(event: str) -> str:
    s = event
    for k, v in PERSON_MAP.items():
        s = s.replace(k, v)
    s = s.replace("___", BLANK)
    return s


def clean_lemma(tok) -> str:
    return tok.lemma_.lower().strip()


def candidate_from_token(tok):
    """Restituisce (surface_lemma, n_tokens) per un nome-testa, includendo un
    eventuale modificatore compound/amod nominale (es. 'credit card', 'hot dog')."""
    head = clean_lemma(tok)
    mods = []
    for c in tok.children:
        if c.dep_ == "compound" and c.pos_ in {"NOUN", "PROPN"} and c.i < tok.i:
            if c.text not in PERSON_NAMES:
                mods.append((c.i, clean_lemma(c)))
    mods.sort()
    parts = [m for _, m in mods] + [head]
    return parts


def extract_from_event(doc):
    """Ritorna lista di dict {lemma, n_tokens, surface, head_dep, governing_verb}."""
    out = []
    for tok in doc:
        if tok.pos_ not in {"NOUN", "PROPN"}:
            continue
        if tok.text in PERSON_NAMES:
            continue
        # i compound sono assorbiti nella testa
        if tok.dep_ == "compound" and tok.head.pos_ in {"NOUN", "PROPN"}:
            continue
        # possessivi di persona ("Alex's dad"): la testa è 'dad', filtrata dopo
        parts = candidate_from_token(tok)
        lemma = "_".join(parts)
        verb = None
        h = tok
        while h.head.i != h.i:
            h = h.head
            if h.pos_ == "VERB":
                verb = h.lemma_.lower()
                break
        out.append({
            "lemma": lemma,
            "n_tokens": len(parts),
            "head": clean_lemma(tok),
            "pos": tok.pos_,
            "dep": tok.dep_,
            "verb": verb,
        })
    return out

In [6]:
nlp = spacy.load(SPACY_MODEL, disable=["ner"])
pre = [preprocess(e) for e in events["event"]]

In [7]:
mentions = []
reasons = Counter()
for event, doc in zip(events["event"], nlp.pipe(pre, batch_size=256)):
    print('EVENT: ' + event)
    print('DOC: ')
    print(doc)
    for c in extract_from_event(doc):
        c["event"] = event
        # --- filtri strutturali, con contatore per il funnel ---
        if c["pos"] == "PROPN":
            reasons["scartato: nome proprio"] += 1
            continue
        if c["head"] in STOP_HEADS:
            reasons["scartato: testa generica/persona/astratta"] += 1
            continue
        if c["n_tokens"] > MAX_TOKENS:
            reasons["scartato: >2 token"] += 1
            continue
        if not re.fullmatch(r"[a-z]+(_[a-z]+)?", c["lemma"]):
            reasons["scartato: caratteri non alfabetici"] += 1
            continue
        if len(c["head"]) < 3:
            reasons["scartato: testa troppo corta"] += 1
            continue
        mentions.append(c)

m = pd.DataFrame(mentions)
funnel["mentions_kept"] = int(len(m))
funnel["discard_reasons"] = dict(reasons)

EVENT: PersonX about to get married
DOC: 
Alex about to get married
EVENT: PersonX beats PersonX's dad
DOC: 
Alex beats Alex's dad
EVENT: PersonX buys a house
DOC: 
Alex buys a house
EVENT: PersonX buys a toy
DOC: 
Alex buys a toy
EVENT: PersonX buys ___ a new one
DOC: 
Alex buys something a new one
EVENT: PersonX buys ___ PersonY needed
DOC: 
Alex buys something Sam needed
EVENT: PersonX burns to a crisp
DOC: 
Alex burns to a crisp
EVENT: PersonX buys a lottery ticket
DOC: 
Alex buys a lottery ticket
EVENT: PersonX buys a suit
DOC: 
Alex buys a suit
EVENT: PersonX accidentally dropped ___ on the ground
DOC: 
Alex accidentally dropped something on the ground
EVENT: PersonX burns PersonX's arm
DOC: 
Alex burns Alex's arm
EVENT: PersonX attacks the enemy 's ___
DOC: 
Alex attacks the enemy 's something
EVENT: PersonX buys coffee
DOC: 
Alex buys coffee
EVENT: PersonX almost hit ___
DOC: 
Alex almost hit something
EVENT: PersonX asks PersonY to take
DOC: 
Alex asks Sam to take
EVENT: Perso

In [8]:
min_events = 1

# --- aggregazione per entità --------------------------------------------
verbs = defaultdict(Counter)
for lemma, v in zip(m["lemma"], m["verb"]):
    if isinstance(v, str) and v:
        verbs[lemma][v] += 1
g = m.groupby("lemma")
ents = pd.DataFrame({
    "entity": g.size().index,
    "n_mentions": g.size().values,
    "n_events": g["event"].nunique().values,
    "n_tokens": g["n_tokens"].first().values,
    "example_events": g["event"].apply(lambda s: " | ".join(s.drop_duplicates().head(3))).values,
})
ents["top_verbs"] = ents["entity"].map(
    lambda e: ",".join(f"{v}:{n}" for v, n in verbs[e].most_common(3)))
ents["source"] = "f_atomic_event"
ents = ents[ents["n_events"] >= min_events]
ents = ents.sort_values(["n_events", "entity"], ascending=[False, True])

funnel["distinct_entities"] = int(len(ents))
funnel["entities_in_1_event"] = int((ents["n_events"] == 1).sum())

ents.to_csv(objects_path / "event_entities.csv", index=False)
m[["event", "lemma", "head", "dep", "verb"]].rename(columns={"lemma": "entity"}) \
        .to_csv(objects_path / "event_entity_mentions.csv", index=False)
(results_path / "extraction_funnel.json").write_text(json.dumps(funnel, indent=2))
print(json.dumps(funnel, indent=2))
print(ents.head(40).to_string(index=False))

{
  "event_rows": 9100,
  "unique_events": 4648,
  "mentions_kept": 3318,
  "discard_reasons": {
    "scartato: testa generica/persona/astratta": 449,
    "scartato: nome proprio": 34,
    "scartato: testa troppo corta": 16,
    "scartato: caratteri non alfabetici": 2
  },
  "distinct_entities": 1066,
  "entities_in_1_event": 574
}
     entity  n_mentions  n_events  n_tokens                                                                                                                      example_events                top_verbs         source
      money         178       178         1                                                     PersonX borrows some money | PersonX borrows the money | PersonX begs for money   have:19,save:17,get:16 f_atomic_event
        car          62        62         1                                              PersonX buys a new car | PersonX buys PersonY a car | PersonX buys PersonX's first car      get:14,buy:13,hit:9 f_atomic_event
       hand       

In [17]:
"""
Aggiunge a event_entities.csv una pre-classificazione "oggetto fisico sì/forse/no"
basata sulla categoria WordNet (lexname) dei sensi del nome.

In WordNet i sensi sono ordinati per frequenza d'uso, e il primo senso è
un buon proxy del senso principale.
"""

def classify(entity: str) -> dict:
    ss = wn.synsets(entity, pos="n")  # gli spazi sono già underscore
    if not ss:
        return {"wn_n_senses": 0, "wn_first_lexname": None,
                "wn_share_physical": None, "physical": "maybe",
                "physical_reason": "assente in WordNet"}
    lex = [s.lexname() for s in ss]
    first = lex[0]
    share = sum(l in PHYS for l in lex) / len(lex)
    if first in PHYS:
        label, why = "yes", f" reason: {first}"
    elif share > SHARE_THRESHOLD or first in AMBIGUOUS:
        label, why = "maybe", f"reason: {first}, physical share: {share:.2f}"
    else:
        label, why = "no", f"reason: {first}, no physical share"
    return {"wn_n_senses": len(ss), "wn_first_lexname": first,
            "wn_share_physical": round(share, 2), "physical": label,
            "physical_reason": why}

In [18]:
feats = []
feats = pd.DataFrame([classify(e) for e in ents["entity"]])
res = pd.concat([ents.reset_index(drop=True), feats], axis=1)

In [19]:
res.to_csv(final_entity_path, index=False)
print(res["physical"].value_counts())

physical
no       396
maybe    394
yes      276
Name: count, dtype: int64
